# Lab 3 · การวางแผนและการให้เหตุผล

Lab นี้มีสองช่วง ช่วงแรกเทียบ **classical planning** (search บนสถานะ) กับการให้ LLM วางแผน ช่วงที่สองใช้รูปแบบการวางแผนของ agent สามแบบกับงานจริง: ตรวจสัญญา 3 ฉบับเทียบกับ policy ของบริษัท 8 ข้อ แล้ววัดว่าแบบไหนแม่นที่สุด และแลกด้วยต้นทุนเท่าไร

| ส่วน | สิ่งที่ทำ | เวลา |
|---|---|---|
| 1 | Classical planning ด้วย breadth-first search | 10 นาที |
| 2 | LLM เป็นผู้วางแผน และตัวตรวจแผน | 10 นาที |
| 3 | โจทย์ตรวจสัญญา และวิธีถามครั้งเดียว | 5 นาที |
| 4 | ให้เหตุผลก่อนตัดสิน | 8 นาที |
| 5 | Plan-and-Execute ด้วย LangGraph | 15 นาที |
| 6 | Reflection: ร่างแล้วตรวจ | 10 นาที |
| 7 | เปรียบเทียบ | 5 นาที |
| 8 | Stretch: ReAct และ self-consistency | ถ้ามีเวลา |

ส่วนที่ 3–6 เรียก LLM หลายสิบครั้ง เวลาที่เขียนไว้ในแต่ละ cell มาจากเครื่องทดสอบ เครื่องของคุณอาจช้ากว่า

มองหา `# TODO` ใน code ก่อนเริ่มให้บันทึกไฟล์นี้เป็น `lab3_planning.mine.ipynb`

## 1. Classical planning ด้วย breadth-first search (10 นาที)

ในแบบของ AIMA บทที่ 10 ปัญหาการวางแผนประกอบด้วย **สถานะ** (ชุดของข้อเท็จจริงที่เป็นจริง), **action** ที่มีเงื่อนไขก่อนทำ (`pre`) และผลที่เพิ่ม (`add`) หรือลบ (`del`) ข้อเท็จจริง, สถานะเริ่มต้น และ **เป้าหมาย** Planner ค้นหาลำดับ action ที่พาจากสถานะเริ่มต้นไปถึงเป้าหมาย

โดเมนด้านล่างคือขั้นตอนการอนุมัติใบแจ้งหนี้ สังเกตว่า `submit_for_approval` **ลบ** `editable`: ส่งอนุมัติแล้วจะแก้ยอดไม่ได้

**งานของคุณ:** เขียน `applicable` (action ทำได้เมื่อ `pre` เป็นจริงทั้งหมดในสถานะนั้น) และ `apply` (ลบ `del` แล้วเพิ่ม `add`)

In [ ]:
import time
from collections import deque

ACTIONS = {
    "extract_fields":      {"pre": {"have_text"},                   "add": {"have_fields"},    "del": set()},
    "check_total":         {"pre": {"have_fields"},                 "add": {"total_checked"},  "del": set()},
    "correct_amount":      {"pre": {"have_fields", "editable"},     "add": {"amount_correct"}, "del": set()},
    "submit_for_approval": {"pre": {"total_checked"},               "add": {"submitted"},      "del": {"editable"}},
    "reopen":              {"pre": {"submitted"},                   "add": {"editable"},       "del": {"submitted"}},
    "approve":             {"pre": {"submitted", "amount_correct"}, "add": {"approved"},       "del": set()},
}
START, GOAL = {"have_text", "editable"}, {"approved"}


def applicable(action, state, actions=ACTIONS):
    # TODO: return True if every precondition of the action is in the state
    return True


def apply(action, state, actions=ACTIONS):
    # TODO: return a new frozenset: the state minus the action's "del" facts, plus its "add" facts
    return frozenset(state)


def bfs_plan(start, goal, actions=ACTIONS):
    """Breadth-first search over states. Returns (shortest plan, number of states explored)."""
    start = frozenset(start)
    frontier, seen = deque([(start, [])]), {start}
    while frontier:
        state, plan = frontier.popleft()
        if goal <= state:
            return plan, len(seen)
        for action in actions:
            if applicable(action, state, actions):
                nxt = apply(action, state, actions)
                if nxt not in seen:
                    seen.add(nxt)
                    frontier.append((nxt, plan + [action]))
    return None, len(seen)


def validate(plan, start, goal, actions=ACTIONS):
    """Check a plan step by step. Returns 'valid' or a message saying where it fails."""
    state = frozenset(start)
    for i, action in enumerate(plan, 1):
        if action not in actions:
            return f"step {i}: there is no action called {action}"
        if not applicable(action, state, actions):
            return f"step {i} ({action}) cannot run: it needs {sorted(actions[action]['pre'] - state)}, which is not true at that point"
        state = apply(action, state, actions)
    return "valid" if goal <= state else f"the goal is not reached: {sorted(goal - state)} is still missing"


plan, explored = bfs_plan(START, GOAL)
if plan is None:
    print(f"no plan found after exploring {explored} states (have you written applicable and apply?)")
else:
    print("plan:", plan, f"\nexplored {explored} states\ncheck:", validate(plan, START, GOAL))
print("a wrong order:", validate(["extract_fields", "check_total", "submit_for_approval", "correct_amount", "approve"], START, GOAL))

## 2. LLM เป็นผู้วางแผน และตัวตรวจแผน (10 นาที)

ตอนนี้ให้ LLM วางแผนแทน search โดยอธิบายโดเมนเป็นข้อความ เราตรวจแผนที่ได้ด้วย `validate` ถ้าผิด ส่งข้อความบอกจุดผิดกลับไปให้แก้ ไม่เกิน 4 รอบ รูปแบบนี้คือการจับคู่ LLM กับตัวตรวจภายนอก (บางครั้งเรียกว่า LLM-Modulo)

In [ ]:
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field

from agentlab import get_llm, release_model

llm = get_llm()


def describe(actions):
    lines = []
    for name, a in actions.items():
        line = f"- {name}: needs {sorted(a['pre'])}; makes true {sorted(a['add'])}"
        if a["del"]:
            line += f"; makes false {sorted(a['del'])}"
        lines.append(line)
    return "\n".join(lines)


def llm_plan(start, goal, actions=ACTIONS, rounds=4):
    """Ask the LLM for a plan, check it, and send back the error until it is valid or rounds run out."""
    class Plan(BaseModel):
        steps: list[Literal[tuple(actions)]] = Field(description="The actions in the order to perform them")

    messages = [
        SystemMessage("You plan a sequence of actions. An action can run only when everything it needs is true at that moment. "
                      "Its results become true, and anything it makes false stops being true."),
        HumanMessage(f"Actions:\n{describe(actions)}\n\nTrue now: {sorted(start)}\nGoal: {sorted(goal)}\n\nGive the shortest plan."),
    ]
    for attempt in range(1, rounds + 1):
        plan = llm.with_structured_output(Plan).invoke(messages)
        verdict = validate(plan.steps, start, goal, actions)
        print(f"  round {attempt}: {plan.steps} -> {verdict}")
        if verdict == "valid":
            return plan.steps
        messages += [AIMessage(str(plan.steps)), HumanMessage(f"That plan is invalid: {verdict}. Give a corrected plan.")]
    return None


print("LLM planner:")
llm_plan(START, GOAL)

LLM น่าจะวางแผนถูกในรอบแรก แต่มันคิดจากเงื่อนไขจริงๆ หรือแค่คุ้นกับชื่ออย่าง `submit_for_approval` และ `approve`

Cell ด้านล่างเปลี่ยน **ชื่อ** ของทุก action และทุกข้อเท็จจริงเป็นคำที่ไม่มีความหมาย โครงสร้าง (เงื่อนไขและผล) เหมือนเดิมทุกประการ แล้วให้ทั้ง search และ LLM วางแผนใหม่ (ราว 30 วินาที)

In [ ]:
NONSENSE = {"extract_fields": "plim", "check_total": "dorve", "correct_amount": "sneb", "submit_for_approval": "quarn",
            "reopen": "tobble", "approve": "wexit", "have_text": "zorg", "have_fields": "blick", "total_checked": "frap",
            "editable": "glent", "amount_correct": "mibber", "submitted": "harl", "approved": "yuxo"}
rename = lambda names: {NONSENSE[n] for n in names}
MYSTERY = {NONSENSE[name]: {part: rename(facts) for part, facts in a.items()} for name, a in ACTIONS.items()}
M_START, M_GOAL = rename(START), rename(GOAL)

plan, explored = bfs_plan(M_START, M_GOAL, MYSTERY)
print("search:", plan, "->", validate(plan, M_START, M_GOAL, MYSTERY) if plan else "no plan found")
print("LLM planner:")
llm_plan(M_START, M_GOAL, MYSTERY)
release_model()

ในการทดสอบ LLM ล้มเหลวทั้ง 4 รอบกับโดเมนที่ชื่อไม่มีความหมาย แม้จะได้คำบอกจุดผิดทุกรอบ ส่วน search ได้แผนเดิมทันที ผลของคุณอาจต่างไปเล็กน้อย ประเด็นคือ **LLM วางแผนจากความคุ้นเคยกับภาษา** ส่วน classical planner วางแผนจากกฎ และรับประกันว่าแผนถูกภายใต้ model ของโดเมน ระบบที่ดีใช้ทั้งสองอย่าง: LLM เสนอ ส่วน code ตรวจ

## 3. โจทย์ตรวจสัญญา และวิธีถามครั้งเดียว (5 นาที)

Policy ของบริษัทมี 8 ข้อ บางข้อใช้กับสัญญาทุกฉบับ บางข้อใช้เฉพาะสัญญาบริการ การซื้อสินค้า สัญญาที่มูลค่าเกิน 500,000 บาท หรือสัญญาที่ผู้รับจ้างประมวลผลข้อมูลส่วนบุคคล เราตรวจสัญญา 3 ฉบับ (`CLASS_SET`) และวัดผลด้วย precision (สิ่งที่รายงานถูกกี่ส่วน) กับ recall (สิ่งที่ผิดจริงถูกพบกี่ส่วน)

วิธีแรกคือส่ง policy ทั้งหมดกับสัญญาไปพร้อมกัน แล้วถามครั้งเดียว (ราว 15 วินาที)

In [ ]:
import pandas as pd

from agentlab.contracts import CLASS_SET, RULES, evaluate, load_contracts, load_labels, policy_text, score_line

contracts, labels = load_contracts(), load_labels()
print(policy_text())
print("\n--- c1 ---\n" + contracts["c1"])

RuleId = Literal["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8"]
REVIEWER = "You review contracts against the company contract policy. A rule counts only if it applies to this kind of contract."


class Violations(BaseModel):
    violated: list[RuleId] = Field(description="IDs of the policy rules this contract breaks; empty if it breaks none")


results, calls = {}, {"n": 0}


def ask(schema, system, user, model=None):
    """One structured LLM call, counted. Retried once: a small model sometimes fails to finish the JSON."""
    chain = (model or llm).with_structured_output(schema)
    messages = [SystemMessage(system), HumanMessage(user)]
    calls["n"] += 1
    try:
        return chain.invoke(messages)
    except Exception:
        calls["n"] += 1
        return chain.invoke(messages)


def run_pattern(name, review):
    """Run review(contract_id) -> set of rule IDs on CLASS_SET, then score it."""
    calls["n"], start = 0, time.time()
    found = {cid: review(cid) for cid in CLASS_SET}
    release_model()
    table = evaluate(found, CLASS_SET)
    results[name] = {**score_line(name, table), "llm_calls": calls["n"], "seconds": round(time.time() - start)}
    print(results[name])
    return table


check_llm = get_llm(num_predict=300)  # a short cap: checking one rule needs a sentence, not a page


def check_rule(cid, rid, schema, system="You check one contract against one policy rule."):
    """Ask whether one contract satisfies one rule; schema decides what the answer looks like (parts 4 and 5)."""
    return ask(schema, system, f"Rule {rid}: {RULES[rid]['text']}\n\nContract:\n{contracts[cid]}\n\nDoes the contract satisfy this rule?", model=check_llm)


def single_call(cid):
    out = ask(Violations, REVIEWER, f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nWhich rules does the contract break?")
    return set(out.violated)


run_pattern("single call", single_call)

## 4. ให้เหตุผลก่อนตัดสิน (8 นาที)

ใน structured output model สร้าง field ตามลำดับใน schema ถ้า field คำตอบ (`complies`) มาก่อน model ต้องตัดสินก่อนได้คิด ถ้า field เหตุผลมาก่อน model ได้ "คิดออกมา" ก่อนตัดสิน ซึ่งคือ chain-of-thought ในรูปของ schema

Cell ด้านล่างตรวจ 4 จุดที่ยากด้วย schema สองแบบ แล้วนับว่าแบบไหนถูกกว่า (ราว 1 นาที)

In [ ]:
class AnswerFirst(BaseModel):
    complies: bool = Field(description="true if the contract satisfies the rule")
    reasoning: str = Field(description="One short sentence explaining why")


class ReasonFirst(BaseModel):
    reasoning: str = Field(description="One short sentence: what the contract says on this point, and whether that meets the rule")
    complies: bool = Field(description="true if the contract satisfies the rule")


hard_cases = [("c5", "R2"), ("c5", "R7"), ("c3", "R7"), ("c1", "R2")]
rows = []
for cid, rid in hard_cases:
    truth = rid not in labels.loc[cid, "violated_rules"]
    for schema in (AnswerFirst, ReasonFirst):
        out = check_rule(cid, rid, schema)
        rows.append({"case": f"{cid}/{rid}", "schema": schema.__name__, "correct": out.complies == truth, "reasoning": out.reasoning[:90]})
release_model()
table = pd.DataFrame(rows)
print(table.groupby("schema")["correct"].sum().to_string())
table

อ่านคอลัมน์ `reasoning` ของแถวที่ผิด บางครั้ง model เขียนเหตุผลที่ถูกแต่ตอบตรงข้าม เพราะมันตัดสินไปก่อนแล้ว บนเครื่องทดสอบ แบบตัดสินก่อนถูก 1 จาก 4 และแบบให้เหตุผลก่อนถูก 4 จาก 4 ผลของคุณอาจต่างไป เพราะ 4 กรณีเป็นตัวอย่างที่เล็ก

**Thinking (ไม่บังคับ ใช้เวลาราว 1.5 นาที):** model ของวิชานี้ "คิด" ก่อนตอบได้ ถ้าเปิดโหมดนี้ (Module 0 และ 1) cell ด้านล่างให้ schema แบบตัดสินก่อนลองตรวจ c5/R2 อีกครั้งโดยเปิด thinking สังเกตเวลาและจำนวน token

In [ ]:
thinking_llm = get_llm(reasoning=True, num_predict=4096)  # thinking needs room: it can use a thousand tokens
start = time.time()
raw = thinking_llm.with_structured_output(AnswerFirst, include_raw=True).invoke(
    [SystemMessage("You check one contract against one policy rule."),
     HumanMessage(f"Rule R2: {RULES['R2']['text']}\n\nContract:\n{contracts['c5']}\n\nDoes the contract satisfy this rule?")])
release_model()
print(f"answer: {raw['parsed']}\n{time.time() - start:.0f}s, {raw['raw'].usage_metadata['output_tokens']} output tokens")

## 5. Plan-and-Execute ด้วย LangGraph (15 นาที)

แทนที่จะตรวจทุกข้อในครั้งเดียว เราวางแผนก่อนว่าต้องตรวจข้อไหน แล้วตรวจทีละข้อ:

1. **LLM ดึงข้อเท็จจริง** ของสัญญา: ซื้อสินค้าหรือไม่ เป็นงานบริการหรือไม่ มูลค่า และมีข้อมูลส่วนบุคคลหรือไม่
2. **Code วางแผน** จากข้อเท็จจริง: เลือกเฉพาะกฎที่ใช้กับสัญญานี้
3. **LLM ตรวจทีละกฎ** ด้วย schema แบบให้เหตุผลก่อน

**งานของคุณ (1):** เติมเงื่อนไขใน `plan_rules` สำหรับ R3, R4, R5 และ R8 ตาม scope ในข้อความ policy ด้านบน

In [ ]:
class Facts(BaseModel):
    # Two yes/no questions, not one "service / goods / other" choice: in testing the small model
    # labelled a computer purchase "service" every time with the choice, and got it right with yes/no.
    buys_goods: bool = Field(description="true if the company is buying products such as computers, equipment or materials")
    is_service: bool = Field(description="true if the supplier performs ongoing work such as maintenance, cleaning, consulting or hosting")
    total_value_thb: float = Field(description="Total value of the contract in THB for its whole term")
    processes_personal_data: bool = Field(description="true if the supplier will handle personal data, such as customers' names or phone numbers")


def plan_rules(facts: Facts) -> list[str]:
    """The plan: which policy rules apply to a contract with these facts."""
    rules = ["R1", "R2", "R6", "R7"]  # these apply to every contract
    # TODO: add R3 if the total value is over 500,000 THB
    # TODO: add R4 if it is a service contract (facts.is_service)
    # TODO: add R5 if the supplier processes personal data
    # TODO: add R8 if it is a contract to buy goods (facts.buys_goods)
    return sorted(rules)


for cid in CLASS_SET:
    facts = ask(Facts, "You extract facts from a contract.", contracts[cid])
    print(cid, facts, "->", plan_rules(facts), "| answer key:", sorted(labels.loc[cid, "applicable_rules"]))
release_model()

ทำไมให้ code วางแผน ไม่ให้ LLM เลือกกฎเอง: ในการทดสอบ เมื่อให้ LLM เลือกกฎที่ใช้บังคับเองจากข้อความ policy มันใส่กฎที่ไม่เกี่ยวมาด้วย เช่น กฎเรื่องการรับประกันสินค้ากับสัญญาบริการ ทำให้รายงานผิดจำนวนมาก การดึงข้อเท็จจริงสามอย่างเป็นงานที่ LLM ทำได้แม่น ส่วนการเทียบกับเงื่อนไขเป็นงานของ code ตามหลักใน Module 1–2

### LangGraph
ตอนนี้ประกอบขั้นตอนเป็น graph ใน LangGraph:

- **State** คือ dict ที่ทุก node อ่านและแก้ (`ReviewState`)
- **Node** คือ function ที่รับ state แล้วคืนส่วนที่เปลี่ยน
- **Edge** บอกว่า node ไหนทำต่อจาก node ไหน และ **conditional edge** เลือก node ถัดไปตาม state

**งานของคุณ (2):** เขียน `route` ให้ไป `check_next_rule` เมื่อยังมีกฎในแผน และไป `report` เมื่อแผนว่างแล้ว จากนั้นรัน cell (ราว 3 นาที)

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class ReviewState(TypedDict, total=False):
    contract_id: str
    facts: Facts
    plan: list[str]          # rules still to check
    checks: dict[str, bool]  # rule -> complies
    violations: list[str]


def extract_facts(state):
    return {"facts": ask(Facts, "You extract facts from a contract.", contracts[state["contract_id"]])}


def make_plan(state):
    return {"plan": plan_rules(state["facts"]), "checks": {}}


class RuleCheck(BaseModel):  # reason first, kept brief: Plan-and-Execute makes many of these calls
    reasoning: str = Field(description="At most 15 words: the relevant clause, and whether it meets the rule")
    complies: bool = Field(description="true if the contract satisfies the rule")


def check_next_rule(state):
    rid, rest = state["plan"][0], state["plan"][1:]
    result = check_rule(state["contract_id"], rid, RuleCheck, "You check one contract against one policy rule. Keep the reasoning brief.")
    return {"plan": rest, "checks": {**state["checks"], rid: result.complies}}


def report(state):
    return {"violations": sorted(rid for rid, ok in state["checks"].items() if not ok)}


def route(state):
    # TODO: return "check_next_rule" if state["plan"] still has rules, otherwise "report"
    return "report"


graph = StateGraph(ReviewState)
graph.add_node("extract_facts", extract_facts)
graph.add_node("make_plan", make_plan)
graph.add_node("check_next_rule", check_next_rule)
graph.add_node("report", report)
graph.add_edge(START, "extract_facts")
graph.add_edge("extract_facts", "make_plan")
graph.add_conditional_edges("make_plan", route, ["check_next_rule", "report"])  # the list names the possible next nodes
graph.add_conditional_edges("check_next_rule", route, ["check_next_rule", "report"])
graph.add_edge("report", END)
reviewer = graph.compile()
print(reviewer.get_graph().draw_mermaid())


def plan_and_execute(cid):
    final = reviewer.invoke({"contract_id": cid}, {"recursion_limit": 30})
    return set(final["violations"])


run_pattern("plan-and-execute", plan_and_execute)

## 6. Reflection: ร่างแล้วตรวจ (10 นาที)

Reflection ใช้สอง call: **ผู้ร่าง** หากฎที่สัญญาฝ่าฝืนพร้อมเหตุผลสั้นๆ แล้ว **ผู้ตรวจ** อ่านร่างนั้นคู่กับสัญญาและ policy ตัดสิ่งที่ผิดออก และเพิ่มสิ่งที่ตกหล่น นี่คือรูปแบบ evaluator-optimizer จาก Module 1

**งานของคุณ:** เขียนคำสั่งของผู้ตรวจใน `REVIEW_INSTRUCTION` ให้บอกชัดว่าเมื่อไรควรเก็บข้อค้นพบไว้ เมื่อไรควรตัดทิ้ง และให้ตรวจหาสิ่งที่ผู้ร่างพลาดด้วย จากนั้นรัน cell (ราว 1.5 นาที)

In [ ]:
class Finding(BaseModel):
    rule_id: RuleId
    reason: str = Field(description="At most 15 words")


class Draft(BaseModel):
    findings: list[Finding] = Field(description="Every rule the contract breaks; empty if none")


# TODO: write the reviewer's instruction
REVIEW_INSTRUCTION = "Review the findings."


def reflection(cid):
    draft = ask(Draft, REVIEWER, f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nWhich rules does the contract break, and why?")
    listed = "\n".join(f"- {f.rule_id}: {f.reason}" for f in draft.findings) or "(none)"
    review = ask(Violations, REVIEW_INSTRUCTION,
                 f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nFirst reviewer's findings:\n{listed}\n\nWhich rules does the contract really break?")
    print(f"  {cid}: draft {sorted(f.rule_id for f in draft.findings)} -> reviewed {sorted(review.violated)}")
    return set(review.violated)


run_pattern("reflection", reflection)

## 7. เปรียบเทียบ (5 นาที)

ตารางด้านล่างรวมผลของทุกรูปแบบที่รันไปแล้ว F1 รวม precision กับ recall เป็นตัวเลขเดียว ดูคู่กับจำนวน LLM call และเวลา ซึ่งเป็นต้นทุน

In [ ]:
pd.DataFrame(results.values()).set_index("pattern")[["precision", "recall", "f1", "fp", "fn", "llm_calls", "seconds"]]

## 8. Stretch

### 8.1 ReAct (ราว 3–5 นาที)
ReAct จาก Lab 2: agent ที่มี tools อ่านสัญญาและ policy แล้วตัดสินเองว่าจะตรวจอะไร ไม่มีแผนล่วงหน้า ในการทดสอบกับสัญญา 5 ฉบับ มันมักหยุดก่อนตรวจครบ precision สูงแต่ recall ต่ำ

### 8.2 Self-consistency
ถามแบบครั้งเดียว 3 ครั้งด้วย temperature 0.8 แล้วเก็บเฉพาะกฎที่ถูกรายงานอย่างน้อย 2 ใน 3 ครั้ง ผลเปลี่ยนไปอย่างไรเทียบกับถามครั้งเดียว

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

from agentlab.contracts import TOOLS as CONTRACT_TOOLS


def react(cid):
    reported = {}

    @tool
    def report_violations(violated: list[RuleId]) -> str:
        """Report the final list of rule IDs this contract breaks (an empty list if none). Call it once, at the end."""
        reported["rules"] = set(violated)
        return "reported"

    agent = create_agent(llm, CONTRACT_TOOLS + [report_violations],
                         system_prompt=REVIEWER + " Read the contract, list the policy rules, check each rule that applies, then call report_violations once.")
    out = agent.invoke({"messages": [HumanMessage(f"Review contract {cid}.")]}, {"recursion_limit": 25})
    calls["n"] += sum(m.type == "ai" for m in out["messages"])  # every AI message is one model call
    return reported.get("rules", set())


run_pattern("ReAct", react)

In [ ]:
voting_llm = get_llm(temperature=0.8)


def self_consistency(cid, samples=3):
    votes = {}
    for _ in range(samples):
        out = ask(Violations, REVIEWER, f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nWhich rules does the contract break?", model=voting_llm)
        for rid in set(out.violated):
            votes[rid] = votes.get(rid, 0) + 1
    return {rid for rid, n in votes.items() if n >= 2}


run_pattern("self-consistency (3 votes)", self_consistency)
pd.DataFrame(results.values()).set_index("pattern")[["precision", "recall", "f1", "fp", "fn", "llm_calls", "seconds"]]

## สรุปและสิ่งที่ต้องส่ง

ตอบคำถาม 4 ข้อนี้ ข้อละ 2–3 ประโยค แล้วส่งพร้อมภาพตารางในส่วนที่ 7

1. ในส่วนที่ 2 ทำไม LLM วางแผนถูกเมื่อชื่อมีความหมาย แต่ผิดเมื่อชื่อไม่มีความหมาย ทั้งที่โครงสร้างเหมือนกัน ข้อนี้บอกอะไรเกี่ยวกับการให้ LLM วางแผนในงานจริง
2. ในส่วนที่ 4 ทำไมลำดับของ field ใน schema จึงเปลี่ยนความถูกต้องได้
3. จากตารางในส่วนที่ 7 คุณจะเลือกรูปแบบไหนสำหรับตรวจสัญญาวันละ 200 ฉบับ และจะเลือกรูปแบบไหนถ้าการปล่อยสัญญาที่ผิดผ่านไปมีราคาแพงมาก
4. Plan-and-Execute ในส่วนที่ 5 ให้ LLM ทำอะไร และให้ code ทำอะไร ทำไมจึงแบ่งแบบนี้

**จาก notebook สู่โปรแกรม:** `contract_review.py` ในโฟลเดอร์เดียวกันรันรูปแบบเหล่านี้กับสัญญาครบ 5 ฉบับ และบันทึกเหตุผลของการตรวจทุกข้อ ดูวิธีรันใน `README.md` และคำอธิบายทีละส่วนใน `contract_review.th.md`

**Project milestone 3:** ร่างแผนการทำงานของ agent ในโปรเจกต์ของกลุ่มเป็น graph (node และ edge) ระบุว่าขั้นไหนให้ LLM ทำ ขั้นไหนให้ code ทำ และจุดไหนที่ควรมีการตรวจหรือ reflection